In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TextClassificationPipeline
import joblib

In [49]:
tokenizer = AutoTokenizer.from_pretrained("finiteautomata/bertweet-base-sentiment-analysis")
model = AutoModelForSequenceClassification.from_pretrained(
    "finiteautomata/bertweet-base-sentiment-analysis",
    id2label = {0: 'Negative', 1: 'Neutral', 2: 'Positive'},
    label2id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7172.56it/s]


In [3]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

df = pd.read_csv("processed_sentiment_data.csv")

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

def tokenize_fn(batch):
  return tokenizer(
      batch["processed_text"],
      truncation = True,
      max_length= 100,
      padding =False,

  )
tokenized_datasets = dataset.map(tokenize_fn, batched = True)
tokenized_datasets = tokenized_datasets.remove_columns(["processed_text"])
tokenized_datasets = tokenized_datasets.rename_column("sentiment", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_ds = tokenized_datasets["train"].shuffle(seed=42).select(range(5000))
eval_ds  = tokenized_datasets["test"].shuffle(seed=42).select(range(1000))

Map: 100%|██████████| 21693/21693 [00:15<00:00, 1419.78 examples/s]


In [4]:
!pip install -q evaluate scikit-learn
!pip install -q -U torchvision


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import evaluate
from transformers import TrainingArguments, Trainer

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }
training_args = TrainingArguments(
    output_dir="./roberta-sentiment-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    #per_device_train_batch_size controls training batch size. per_device_eval_batch_size evaluation batch size.
    learning_rate=.00001,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01, #this adds a L2 penalty
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True, #half precision
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
C:\Users\sungj\Documents\GitHub\Atlantic-Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
metrics = trainer.evaluate()
print(metrics)
from sklearn.metrics import classification_report
preds_output = trainer.predict(eval_ds)
y_pred = np.argmax(preds_output.predictions, axis =-1)
y_true = preds_output.label_ids
print(classification_report(y_true, y_pred))

In [ ]:
trainer.save_model("roberta_sentiment_model")
tokenizer.save_pretrained("roberta_sentiment_model")

In [23]:
#I didnt now how to do the actual predictions part and I wanted to find an easy way to use the tokenizer and model. Using this text classification pipeline makes it easy to combine it, instead of making a predict method.
#https://discuss.huggingface.co/t/i-have-trained-my-classifier-now-how-do-i-do-predictions/3625
sent_tokenizer = AutoTokenizer.from_pretrained("roberta_sentiment_model")
sent_model = AutoModelForSequenceClassification.from_pretrained("roberta_sentiment_model")

full_model = TextClassificationPipeline(
    model = sent_model,
    tokenizer = sent_tokenizer,
    return_all_scores = True
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1593.57it/s]


In [59]:
from logreg_class import LogRegTfidfModel

def predict(text):
    model = LogRegTfidfModel()
    model.predict(text)
    pred = full_model(text)
    print(f"** Roberta Model **\nPredicted Sentiment: {pred[0]['label']}\nConfidence: %{(pred[0]['score'] * 100):.2f}")

In [ ]:
predict("The house is in okay condition, but it could be better")